# Tutorial 1: The Basic Tools of the Deep Life Sciences
Welcome to DeepChem's introductory tutorial for the deep life sciences. This series of notebooks is a step-by-step guide for you to get to know the new tools and techniques needed to do deep learning for the life sciences. We'll start from the basics, assuming that you're new to machine learning and the life sciences, and build up a repertoire of tools and techniques that you can use to do meaningful work in the life sciences.

**Scope:** This tutorial will encompass both the machine learning and data handling needed to build systems for the deep life sciences.

## Colab

This tutorial and the rest in the sequences are designed to be done in Google colab. If you'd like to open this notebook in colab, you can use the following link.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepchem/deepchem/blob/master/examples/tutorials/01_The_Basic_Tools_of_the_Deep_Life_Sciences.ipynb)


## Why do the DeepChem Tutorial?

**1) Career Advancement:** Applying AI in the life sciences is a booming
industry at present. There are a host of newly funded startups and initiatives
at large pharmaceutical and biotech companies centered around AI. Learning and
mastering DeepChem will bring you to the forefront of this field and will
prepare you to enter a career in this field.

**2) Humanitarian Considerations:** Disease is the oldest cause of human
suffering. From the dawn of human civilization, humans have suffered from pathogens,
cancers, and neurological conditions. One of the greatest achievements of
the last few centuries has been the development of effective treatments for
many diseases. By mastering the skills in this tutorial, you will be able to
stand on the shoulders of the giants of the past to help develop new
medicine.

**3) Lowering the Cost of Medicine:** The art of developing new medicine is
currently an elite skill that can only be practiced by a small core of expert
practitioners. By enabling the growth of open source tools for drug discovery,
you can help democratize these skills and open up drug discovery to more
competition. Increased competition can help drive down the cost of medicine.

## Getting Extra Credit
If you're excited about DeepChem and want to get more involved, there are some things that you can do right now:

* Star DeepChem on GitHub! - https://github.com/deepchem/deepchem
* Join the DeepChem forums and introduce yourself! - https://forum.deepchem.io
* Say hi on the DeepChem Discord - https://discord.gg/cGzwCdrUqS
* Make a YouTube video teaching the contents of this notebook.


## Prerequisites

This tutorial sequence will assume some basic familiarity with the Python data science ecosystem. We will assume that you have familiarity with libraries such as Numpy, Pandas, and TensorFlow. We'll provide some brief refreshers on basics through the tutorial so don't worry if you're not an expert.

## Setup

The first step is to get DeepChem up and running. We recommend using Google Colab to work through this tutorial series. You'll need to run the following cell to get DeepChem installed on your colab notebook. Installation usually takes a minute or two. If Colab asks you to restart the session after the install cell, do so (**Runtime → Restart session**) and then continue with the next cell; there is no need to run the install cell again.

In [1]:
# DeepChem itself, RDKit (reads molecules) and tf_keras. DeepChem's TensorFlow models
# (such as GraphConvModel, used below) need the legacy Keras 2 API, which recent
# TensorFlow versions only provide through the separate "tf_keras" package.
%pip install -q deepchem rdkit tf_keras

# Colab normally ships with TensorFlow; install it only if it is missing.
import importlib.util
if importlib.util.find_spec("tensorflow") is None:
    print("TensorFlow not found - installing it ...")
    %pip install -q tensorflow

You can of course run this tutorial locally if you prefer. In that case, install the same packages in your own Python environment (for example `pip install deepchem rdkit tf_keras tensorflow`) instead of running the cell above. In either case, we can now import the `deepchem` package to play with.

Two details about the import cell below:

* It sets the environment variable `TF_USE_LEGACY_KERAS=1` so that TensorFlow uses Keras 2, which is what DeepChem's TensorFlow models expect. This **must happen before TensorFlow or DeepChem is imported**. If you ever see an error about `keras.optimizers.legacy` or Keras 3, restart the session (**Runtime → Restart session**) and run the import cell first.
* DeepChem may print warnings about optional packages (e.g. PyTorch or JAX models) that are not installed. They are harmless for this tutorial.

In [2]:
import os, sys, warnings

# Use Keras 2 (tf_keras) inside TensorFlow. Must be set BEFORE importing tensorflow/deepchem.
os.environ["TF_USE_LEGACY_KERAS"] = "1"
if "tensorflow" in sys.modules:
    print("WARNING: TensorFlow was already imported in this session. "
          "Use Runtime > Restart session and run this cell before any other TensorFlow code.")

# Silence harmless warnings so the real output stays readable (set BEFORE importing deepchem):
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")                               # TensorFlow C++ info messages
warnings.filterwarnings("ignore", message=".*pkg_resources.*")                   # deprecation notice from a dependency
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")                                                   # RDKit's repeated "GetValence" deprecation lines

import deepchem as dc
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

print("DeepChem", dc.__version__, "| TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none (a CPU runtime is fine for this tutorial)")

# DeepChem silently leaves out its TensorFlow models if TensorFlow cannot be imported;
# fail early with a clear message instead of a confusing error later.
if not hasattr(dc.models, "GraphConvModel"):
    raise ImportError("dc.models.GraphConvModel is unavailable: DeepChem could not load its TensorFlow "
                      "models. Check the messages above, restart the session and re-run the install and import cells.")

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.
Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.


DeepChem 2.5.0 | TensorFlow 2.21.0
GPU: none (a CPU runtime is fine for this tutorial)


# Training a Model with DeepChem: A First Example

Deep learning can be used to solve many sorts of problems, but the basic workflow is usually the same.  Here are the typical steps you follow.

1. Select the data set you will train your model on (or create a new data set if there isn't an existing suitable one).
2. Create the model.
3. Train the model on the data.
4. Evaluate the model on an independent test set to see how well it works.
5. Use the model to make predictions about new data.

With DeepChem, each of these steps can be as little as one or two lines of Python code.  In this tutorial we will walk through a basic example showing the complete workflow to solve a real world scientific problem.

The problem we will solve is predicting the solubility of small molecules given their chemical formulas.  This is a very important property in drug development: if a proposed drug isn't soluble enough, you probably won't be able to get enough into the patient's bloodstream to have a therapeutic effect.  The first thing we need is a data set of measured solubilities for real molecules.  One of the core components of DeepChem is MoleculeNet, a diverse collection of chemical and molecular data sets.  For this tutorial, we can use the Delaney solubility data set.

In [3]:
tasks, datasets, transformers = dc.molnet.load_delaney(featurizer='GraphConv')
train_dataset, valid_dataset, test_dataset = datasets

I won't say too much about this code right now.  We will see many similar examples in later tutorials.  There are two details I do want to draw your attention to.  First, notice the `featurizer` argument passed to the `load_delaney()` function.  Molecules can be represented in many ways.  We therefore tell it which representation we want to use, or in more technical language, how to "featurize" the data.  Second, notice that we actually get three different data sets: a training set, a validation set, and a test set.  Each of these serves a different function in the standard deep learning workflow.

Now that we have our data, the next step is to create a model.  We will use a particular kind of model called a "graph convolutional network", or "graphconv" for short.

In [4]:
model = dc.models.GraphConvModel(n_tasks=1, mode='regression', dropout=0.2)

Here again I will not say much about the code.  Later tutorials will give lots more information about `GraphConvModel`, as well as other types of models provided by DeepChem.

We now need to train the model on the data set.  We simply give it the data set and tell it how many epochs of training to perform (that is, how many complete passes through the data to make).

In [5]:
model.fit(train_dataset, nb_epoch=100)

0.11677821159362793

If everything has gone well, we should now have a fully trained model!  But do we?  To find out, we must evaluate the model on the test set.  We do that by selecting an evaluation metric and calling `evaluate()` on the model.  For this example, let's use the squared Pearson correlation coefficient, r<sup>2</sup>, as our metric.  We can evaluate it on both the training set and test set.

In [6]:
metric = dc.metrics.Metric(dc.metrics.pearson_r2_score)
print("Training set score:", model.evaluate(train_dataset, [metric], transformers))
print("Test set score:", model.evaluate(test_dataset, [metric], transformers))

Training set score: {'pearson_r2_score': np.float64(0.9111259284510398)}
Test set score: {'pearson_r2_score': np.float64(0.6224742561601538)}


Notice that it has a higher score on the training set than the test set.  Models usually perform better on the particular data they were trained on than they do on similar but independent data.  This is called "overfitting", and it is the reason it is essential to evaluate your model on an independent test set.

Our model still has respectable performance on the test set. (Training starts from random weights, so your exact numbers will differ a little from run to run.)  For comparison, a model that produced totally random outputs would have an r<sup>2</sup> near 0, while one that made perfect predictions would have an r<sup>2</sup> of 1.  Our model does quite well, so now we can use it to make predictions about other molecules we care about.

Since this is just a tutorial and we don't have any other molecules we specifically want to predict, let's just use the first ten molecules from the test set.  For each one we print out the chemical structure (represented as a SMILES string), the predicted solubility and, for comparison, the measured value.  The Delaney data set stores solubility as log(mol/L), and DeepChem normalises these values before training, so we first convert the model's output back to the original units with `dc.trans.undo_transforms`.

In [7]:
import numpy as np
import pandas as pd

# Predict the whole (small) test set, then undo the normalisation applied to the targets
# so that the numbers are in the original units: log solubility in mol/L.
predictions = dc.trans.undo_transforms(model.predict(test_dataset), transformers)
measured    = dc.trans.undo_transforms(test_dataset.y, transformers)
n_show = 10
pd.DataFrame({
    "SMILES": test_dataset.ids[:n_show],
    "predicted log(mol/L)": np.asarray(predictions).reshape(len(test_dataset), -1)[:n_show, 0].round(2),
    "measured log(mol/L)":  np.asarray(measured).reshape(len(test_dataset), -1)[:n_show, 0].round(2),
})

,SMILES,predicted log(mol/L),measured log(mol/L)
0,c1cc2ccc3cccc4ccc(c1)c2c34,-6.57,-6.18
1,Cc1cc(=O)[nH]c(=S)[nH]1,-0.63,-2.44
2,Oc1ccc(cc1)C2(OC(=O)c3ccccc23)c4ccc(O)cc4,-2.53,-2.90
3,c1ccc2c(c1)cc3ccc4cccc5ccc2c3c45,-6.88,-8.70
4,C1=Cc2cccc3cccc1c23,-5.25,-3.96
5,CC1CO1,0.68,-0.59
6,CCN2c1ccccc1N(C)C(=S)c3cccnc23,-4.01,-4.71
7,CC12CCC3C(CCc4cc(O)ccc34)C2CCC1=O,-5.24,-3.96
8,Cn2cc(c1ccccc1)c(=O)c(c2)c3cccc(c3)C(F)(F)F,-5.14,-4.44
9,ClC(Cl)(Cl)C(NC=O)N1C=CN(C=C1)C(NC=O)C(Cl)(Cl)Cl,-2.93,-4.19


# Congratulations! Time to join the Community!

Congratulations on completing this tutorial notebook! If you enjoyed working through the tutorial, and want to continue working with DeepChem, we encourage you to finish the rest of the tutorials in this series. You can also help the DeepChem community in the following ways:

## Star DeepChem on [GitHub](https://github.com/deepchem/deepchem)
This helps build awareness of the DeepChem project and the tools for open source drug discovery that we're trying to build.

## Join the DeepChem Discord
The DeepChem [Discord](https://discord.gg/cGzwCdrUqS) hosts a number of scientists, developers, and enthusiasts interested in deep learning for the life sciences. Join the conversation!

# Troubleshooting

| Symptom | What to do |
|---|---|
| `ModuleNotFoundError` for `deepchem`, or odd NumPy / binary-compatibility errors right after the install cell | **Runtime → Restart session**, then run the import cell (no need to reinstall). |
| `ImportError: keras.optimizers.legacy is not supported in Keras 3` | TensorFlow was imported before `TF_USE_LEGACY_KERAS` was set, or `tf_keras` is missing. Restart the session, make sure the install cell ran, and run the import cell first. |
| `AttributeError: module 'deepchem.models' has no attribute 'GraphConvModel'` | DeepChem could not load TensorFlow. Read the warnings printed by the import cell; `%pip install tensorflow tf_keras`, restart, and try again. |
| Training is slow | This data set is small, so a CPU runtime is fine. A GPU runtime (**Runtime → Change runtime type**) can speed it up a little. |